# 01.8 — File I/O & Error Handling

**Phase:** 01 — Python Foundations

**Status:** VERIFIED

---

## 1. What Are We Solving?

Real programs read and write files (data, models, logs) and must handle errors gracefully. **File I/O** lets you persist and load data. **Error handling** (`try`/`except`) lets your code fail gracefully instead of crashing.

## 2. Why Does This Matter?

ML workflows constantly read datasets, save models, and load checkpoints. Robust error handling is essential for production code and for debugging.

## 3. Prerequisites

- Unit 01.1 (Python basics)
- Unit 01.2 (Data structures)

## 4. Learning Objectives

By the end of this notebook, you should be able to:
- Read and write text files
- Use `with` statements for safe file handling
- Read and write CSV files
- Use `try`/`except`/`else`/`finally`
- Raise custom exceptions
- Handle common exception types

## 5. Mental Model

File I/O is like a **file cabinet**: you open a drawer (file), read/write, then close it. The `with` statement guarantees the drawer is closed even if something goes wrong.

Error handling is like a **safety net**: `try` attempts the risky operation, `except` catches failures, `finally` always runs cleanup.

## 6. Writing and Reading Text Files

Use `open()` with a mode: `'w'` (write), `'r'` (read), `'a'` (append).

In [1]:
# Write to a text file
import os

filepath = os.path.join(os.getcwd(), "temp_demo.txt")

with open(filepath, "w") as f:
    f.write("Hello, world!\n")
    f.write("Second line\n")
    f.write("Third line\n")

print(f"Wrote to {filepath}")

# Read it back
with open(filepath, "r") as f:
    content = f.read()
print("Content:")
print(content)

# Clean up
os.remove(filepath)
print("Cleaned up temp file.")

Wrote to D:\CODE\complete ml\notebooks\01_python\temp_demo.txt
Content:
Hello, world!
Second line
Third line

Cleaned up temp file.


In [2]:
# Reading line by line
import os
filepath = os.path.join(os.getcwd(), "temp_lines.txt")

with open(filepath, "w") as f:
    for i in range(5):
        f.write(f"Line {i}\n")

with open(filepath, "r") as f:
    for line in f:
        print(line.strip())

os.remove(filepath)

Line 0
Line 1
Line 2
Line 3
Line 4


## 7. The `with` Statement

The `with` statement automatically closes the file, even if an error occurs. This is the recommended way to handle files.

In [3]:
# with statement guarantees cleanup
import os
filepath = os.path.join(os.getcwd(), "temp_with.txt")

with open(filepath, "w") as f:
    f.write("data")
    # file is auto-closed here even if an error occurred

print(f"File closed automatically: {os.path.exists(filepath)}")
os.remove(filepath)

File closed automatically: True


## 8. Reading and Writing CSV

CSV (comma-separated values) is the most common data format.

In [4]:
# CSV with the csv module
import csv
import os

filepath = os.path.join(os.getcwd(), "temp_data.csv")

# Write CSV
with open(filepath, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "age", "city"])
    writer.writerow(["Ada", 36, "London"])
    writer.writerow(["Bob", 25, "NYC"])

# Read CSV
with open(filepath, "r") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

os.remove(filepath)

['name', 'age', 'city']
['Ada', '36', 'London']
['Bob', '25', 'NYC']


## 9. Error Handling: try/except

`try`/`except` catches errors so your program doesn't crash.

In [5]:
# Basic try/except
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Caught ZeroDivisionError: {e}")

print("Program continues after the error.")

Caught ZeroDivisionError: division by zero
Program continues after the error.


In [6]:
# Multiple except clauses
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return "Cannot divide by zero"
    except TypeError:
        return "Both arguments must be numbers"

print(safe_divide(10, 2))    # 5.0
print(safe_divide(10, 0))    # error message
print(safe_divide(10, "x"))  # error message

5.0
Cannot divide by zero
Both arguments must be numbers


## 10. else and finally

- `else` — runs if no exception occurred.
- `finally` — always runs (cleanup).

In [7]:
# else and finally
def process(value):
    try:
        result = 100 / value
    except ZeroDivisionError:
        print("Error: division by zero")
    else:
        print(f"Success: result = {result}")
    finally:
        print("Cleanup: this always runs")

print("--- value=5 ---")
process(5)
print("--- value=0 ---")
process(0)

--- value=5 ---
Success: result = 20.0
Cleanup: this always runs
--- value=0 ---
Error: division by zero
Cleanup: this always runs


## 11. Raising Exceptions

Use `raise` to signal errors in your own code.

In [8]:
# Raising exceptions
def validate_age(age):
    if age < 0:
        raise ValueError("Age cannot be negative")
    if not isinstance(age, int):
        raise TypeError("Age must be an integer")
    return age

try:
    validate_age(-5)
except ValueError as e:
    print(f"Caught ValueError: {e}")

try:
    validate_age("old")
except TypeError as e:
    print(f"Caught TypeError: {e}")

print(f"validate_age(25): {validate_age(25)}")

Caught ValueError: Age cannot be negative
Caught TypeError: '<' not supported between instances of 'str' and 'int'
validate_age(25): 25


## 12. Custom Exceptions

Define your own exception classes for domain-specific errors.

In [9]:
# Custom exceptions
class InsufficientFundsError(Exception):
    pass

class Account:
    def __init__(self, balance):
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFundsError(
                f"Cannot withdraw {amount}, balance is {self.balance}"
            )
        self.balance -= amount
        return self.balance

acc = Account(100)
try:
    acc.withdraw(150)
except InsufficientFundsError as e:
    print(f"Custom error caught: {e}")

print(f"Withdraw 50: balance = {acc.withdraw(50)}")

Custom error caught: Cannot withdraw 150, balance is 100
Withdraw 50: balance = 50


## 13. Experiment: Robust Data Loader

Let's build a function that safely loads a CSV, handling common errors.

In [10]:
# Experiment: robust CSV loader
import pandas as pd
import os

def safe_load_csv(path):
    try:
        df = pd.read_csv(path)
        return df
    except FileNotFoundError:
        print(f"Error: file not found at {path}")
        return None
    except pd.errors.EmptyDataError:
        print(f"Error: file is empty at {path}")
        return None
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None

# Test with a valid file
filepath = os.path.join(os.getcwd(), "temp_valid.csv")
pd.DataFrame({"a": [1, 2], "b": [3, 4]}).to_csv(filepath, index=False)
df = safe_load_csv(filepath)
print(f"Loaded valid file:\n{df}")
os.remove(filepath)

# Test with a missing file
df = safe_load_csv("nonexistent_file.csv")
print(f"Result for missing file: {df}")

Loaded valid file:
   a  b
0  1  3
1  2  4
Error: file not found at nonexistent_file.csv
Result for missing file: None


## 14. Failure Case: File Not Closed

Forgetting to close a file (or not using `with`) can cause resource leaks and data loss.

In [11]:
# BAD: not using with (file may not be closed)
import os
filepath = os.path.join(os.getcwd(), "temp_bad.txt")

f = open(filepath, "w")
f.write("data")
# forgot f.close() - resource leak

# GOOD: use with
with open(filepath, "w") as f:
    f.write("data")
    # auto-closed

print("Always use 'with' to auto-close files.")
os.remove(filepath)

Always use 'with' to auto-close files.


## 15. Debugging: Common Errors

### FileNotFoundError
```python
open("missing.txt")  # FileNotFoundError
```
Fix: check the path or catch the error.

### PermissionError
File is locked or you lack write access.

### UnicodeDecodeError
Wrong encoding. Use `encoding="utf-8"`.

### Bare except
```python
try:
    ...
except:  # BAD - catches everything, hides bugs
    ...
```
Always catch specific exceptions.

## 16. Real-World Considerations

- **Always use `with`** for file handling.
- **Catch specific exceptions**, not bare `except`.
- **Log errors** rather than just printing.
- **Use `encoding="utf-8"`** for text files.
- **Validate inputs** before processing.

## 17. Common Mistakes

- Not using `with`
- Bare `except` clauses
- Swallowing exceptions silently
- Wrong file mode (`'w'` vs `'a'`)
- Not handling encoding

## 18. When NOT to Use

- Don't use manual file I/O when Pandas handles it (`read_csv`, `to_csv`).
- Don't catch exceptions you can't handle — let them propagate.

## 19. Challenge

Write a function `read_numbers(path)` that reads a text file of numbers (one per line) and returns their sum. Handle the case where the file doesn't exist or contains invalid data.

In [12]:
# Challenge: read and sum numbers from a file
import os

def read_numbers(path):
    try:
        with open(path, "r") as f:
            numbers = []
            for line in f:
                line = line.strip()
                if line:
                    try:
                        numbers.append(float(line))
                    except ValueError:
                        print(f"Skipping invalid line: '{line}'")
            return sum(numbers)
    except FileNotFoundError:
        print(f"Error: file not found at {path}")
        return None

# Test
filepath = os.path.join(os.getcwd(), "temp_numbers.txt")
with open(filepath, "w") as f:
    f.write("1\n2\n3\nabc\n4\n")

total = read_numbers(filepath)
print(f"Sum of valid numbers: {total}")
os.remove(filepath)

Skipping invalid line: 'abc'
Sum of valid numbers: 10.0


## Hands-On Practice

### Level 1: Basic
- Write a string to a file and read it back
- Catch a ZeroDivisionError with try/except

### Level 2: Guided
- Read a CSV file line by line and parse manually
- Write a function that validates input and raises ValueError

### Level 3: Independent
- Build a config file reader with defaults and error handling
- Create a retry decorator for network operations

### Level 4: Realistic
- Build a robust data loader that handles missing files, bad format, and encoding errors

### Level 5: Challenge
- Implement a context manager for database connections


## Knowledge Check

1. What is the difference between `except Exception` and bare `except`?
2. When should you use `finally` vs just `try/except`?
3. What is the benefit of using `with open()` over manual `close()`?



## Exit Criteria

- [ ] Can read and write text and CSV files
- [ ] Can use try/except/else/finally properly
- [ ] Can create custom exceptions
- [ ] Can write robust code that handles edge cases
- [ ] Completed Level 3+ practice



## Next Step

-> Unit 01.9: Synthesis — combine all Python skills into a mini project.


## 20. Closed-Book Recall

Without looking back:

1. What does the `with` statement do?
2. What are the file modes `'r'`, `'w'`, `'a'`?
3. What's the difference between `else` and `finally` in try/except?
4. How do you raise a custom exception?
5. Why should you avoid bare `except`?

## 21. Teach-Back Questions

Explain to another person:

- Why is `with` important for file handling?
- What is exception handling?
- When would you define a custom exception?

## 22. Summary

You now understand file I/O and error handling. These are essential for reading data, saving models, and writing robust code.

## 23. Further Experiment

- Write a JSON file with `json.dump` and read it back.
- Use `pathlib.Path` for modern path handling.
- Build a logging setup with `logging` module.

## 24. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: pandas
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```